# 第四章：工具调用 Agent

## 学习目标
- 理解 LLM 工具调用（Tool Calling）的机制
- 用 `@tool` 装饰器定义工具
- 用 `bind_tools` 让模型知道可用工具
- 手动构建 ReAct Agent（理解内部原理）
- 用 `ToolNode` 简化工具执行
- 用 `create_react_agent` 一行创建完整 Agent

## 0. 环境准备

In [ ]:
from importlib.metadata import version
print(f"langgraph 版本: {version('langgraph')}")

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen-plus",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
)

## 1. 什么是工具调用？

LLM 本身只能生成文本。但现代模型支持一种特殊能力：**Tool Calling**——模型不直接回答，而是输出「我要调用某个工具，参数是 XXX」的结构化指令。

流程是：
1. 告诉模型有哪些工具可用（名称、参数、描述）
2. 用户提问
3. 模型决定是否需要调用工具。如果需要，返回工具名和参数
4. **我们的代码**执行工具，把结果返回给模型
5. 模型根据工具结果生成最终回答

关键点：**模型不执行工具，只决定调用什么**。执行由我们的代码完成。

## 2. 定义工具

LangChain 用 `@tool` 装饰器把普通函数变成工具。函数的 docstring 会被发送给模型作为工具描述。

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """两个整数相乘"""
    return a * b

@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气"""
    # 模拟数据，实际项目中会调用天气 API
    weather_data = {
        "北京": "晴天，15°C",
        "上海": "多云，20°C",
        "深圳": "小雨，25°C",
    }
    return weather_data.get(city, f"{city}：暂无数据")

# 查看工具信息
print(f"工具名: {multiply.name}")
print(f"描述: {multiply.description}")
print(f"参数: {multiply.args}")

### @tool 装饰器的要求

| 要求 | 说明 |
|------|------|
| 类型标注 | 参数必须有类型标注（`a: int`），模型需要知道参数类型 |
| docstring | 必须写，这是模型判断何时使用该工具的依据 |
| 返回值 | 任意类型，LangGraph 会转成字符串发回给模型 |

## 3. bind_tools：让模型知道工具的存在

In [ ]:
tools = [multiply, get_weather]

# bind_tools 返回一个新的模型实例，知道这些工具的存在
llm_with_tools = llm.bind_tools(tools)

In [ ]:
# 模型决定调用工具
response = llm_with_tools.invoke("7 乘以 8 等于多少？")

print(f"content: '{response.content}'")
print(f"tool_calls: {response.tool_calls}")

### 刚才发生了什么？

模型没有直接回答 "56"，而是返回了一个 `tool_calls` 列表，告诉我们：「请调用 `multiply` 工具，参数是 `a=7, b=8`」。

`content` 为空（或很少），因为模型选择了调用工具而非直接回答。

In [ ]:
# 不需要工具时，模型正常回答
response = llm_with_tools.invoke("你好")

print(f"content: '{response.content}'")
print(f"tool_calls: {response.tool_calls}")  # 空列表

模型会自己判断是否需要工具。简单问候不需要工具，就直接文本回复。

## 4. 手动构建 ReAct Agent

现在把工具调用放进 LangGraph 的图中。先**手动**构建，理解内部原理。

ReAct 模式的循环：
1. **模型思考** → 决定调用工具还是直接回答
2. 如果调用工具 → **执行工具** → 把结果返回给模型 → 回到步骤 1
3. 如果直接回答 → 结束

In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.messages import HumanMessage, ToolMessage

# 节点 1：调用模型
def call_model(state: MessagesState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# 节点 2：执行工具
def call_tools(state: MessagesState) -> dict:
    last_message = state["messages"][-1]  # 最后一条是 AIMessage（含 tool_calls）
    tool_map = {t.name: t for t in tools}
    
    results = []
    for tool_call in last_message.tool_calls:
        tool_fn = tool_map[tool_call["name"]]
        result = tool_fn.invoke(tool_call["args"])
        results.append(ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"],
        ))
    return {"messages": results}

# 路由：有 tool_calls → 执行工具；没有 → 结束
def should_call_tools(state: MessagesState) -> Literal["tools", "__end__"]:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "__end__"

# 构建图
graph = StateGraph(MessagesState)
graph.add_node("model", call_model)
graph.add_node("tools", call_tools)
graph.add_edge(START, "model")
graph.add_conditional_edges("model", should_call_tools)
graph.add_edge("tools", "model")  # 工具执行完 → 回到模型（循环）

agent = graph.compile()
from IPython.display import Image
Image(agent.get_graph().draw_mermaid_png())

In [ ]:
# 测试：需要工具的问题
for step in agent.stream({
    "messages": [HumanMessage(content="12 乘以 15 等于多少？")],
}):
    for node, update in step.items():
        print(f"--- [{node}] ---")
        for msg in update["messages"]:
            print(f"  {msg.__class__.__name__}: ", end="")
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                print(f"调用工具 {msg.tool_calls}")
            else:
                print(msg.content[:100])
        print()

In [ ]:
# 测试：不需要工具的问题
result = agent.invoke({
    "messages": [HumanMessage(content="你好，你是谁？")],
})
print(result["messages"][-1].content)

### 刚才发生了什么？

手动构建的 ReAct Agent 有两个节点和一个循环：

```
START → model ──(有 tool_calls)──→ tools → model → ...
                 │
                 └──(无 tool_calls)──→ END
```

消息流：
1. `HumanMessage("12 乘以 15")` → 模型
2. 模型返回 `AIMessage(tool_calls=[{multiply, a=12, b=15}])`
3. 我们执行 multiply → `ToolMessage(content="180")`
4. 模型收到工具结果 → 返回 `AIMessage("12 乘以 15 等于 180")`
5. 没有 tool_calls → 结束

这四种消息类型在 Agent 中各有角色：

| 消息类型 | 谁产生 | 作用 |
|---------|--------|------|
| `HumanMessage` | 用户 | 用户输入 |
| `AIMessage` | 模型 | 回答或工具调用指令 |
| `ToolMessage` | 我们的代码 | 工具执行结果，返回给模型 |
| `SystemMessage` | 我们的代码 | 系统提示（可选） |

## 5. ToolNode：内置的工具执行节点

上面手动写的 `call_tools` 函数逻辑很固定（遍历 tool_calls → 执行 → 包装成 ToolMessage）。LangGraph 提供了 `ToolNode` 来替代。

In [ ]:
from langgraph.prebuilt import ToolNode

# 一行替代手动的 call_tools 函数
tool_node = ToolNode(tools)

# 重新构建图，用 ToolNode 替代手动实现
graph = StateGraph(MessagesState)
graph.add_node("model", call_model)     # 同上
graph.add_node("tools", tool_node)       # ToolNode 替代手动实现
graph.add_edge(START, "model")
graph.add_conditional_edges("model", should_call_tools)
graph.add_edge("tools", "model")

agent = graph.compile()

In [ ]:
# 测试：功能和之前一样
result = agent.invoke({
    "messages": [HumanMessage(content="北京今天天气怎么样？")],
})
print(result["messages"][-1].content)

### 对比：手动 vs ToolNode

| | 手动实现 | ToolNode |
|--|---------|----------|
| 代码量 | ~15 行 | 1 行 |
| 错误处理 | 需自己写 | 内置 |
| 适用场景 | 需要自定义执行逻辑 | 标准工具执行 |

大多数场景用 `ToolNode` 即可。

## 6. create_react_agent：一行创建完整 Agent

前面手动建图的过程（model 节点 + tools 节点 + 条件边 + 循环）是固定模式。LangGraph 提供了 `create_react_agent` 直接生成。

In [ ]:
from langgraph.prebuilt import create_react_agent

# 一行创建完整 Agent
agent = create_react_agent(llm, tools)

from IPython.display import Image
Image(agent.get_graph().draw_mermaid_png())

In [ ]:
# 测试工具调用
result = agent.invoke({
    "messages": [HumanMessage(content="帮我算一下 99 乘以 77")],
})
print(result["messages"][-1].content)

In [ ]:
# 测试多工具场景
for step in agent.stream({
    "messages": [HumanMessage(content="上海天气怎么样？另外帮我算 25 乘以 4")],
}):
    for node, update in step.items():
        print(f"--- [{node}] ---")
        for msg in update["messages"]:
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  调用: {tc['name']}({tc['args']})")
            else:
                print(f"  {msg.__class__.__name__}: {msg.content[:100]}")
        print()

### create_react_agent 做了什么？

它帮你完成了前面手动做的所有事：
1. `llm.bind_tools(tools)`
2. 创建 model 节点（调用模型）
3. 创建 tools 节点（`ToolNode`）
4. 添加条件边（有 tool_calls → tools，否则 → END）
5. 添加循环边（tools → model）

等价于我们手动构建的图，只是一行搞定。

## 7. 带系统提示的 Agent

`create_react_agent` 支持 `prompt` 参数来设定 Agent 的身份和行为。

In [ ]:
# 带系统提示的 Agent
agent = create_react_agent(
    llm,
    tools,
    prompt="你是一个数学助手。对于数学计算，必须使用 multiply 工具，不要心算。对于非数学问题，礼貌拒绝。",
)

result = agent.invoke({
    "messages": [HumanMessage(content="13 乘以 17")],
})
print(result["messages"][-1].content)

In [ ]:
# 测试拒绝非数学问题
result = agent.invoke({
    "messages": [HumanMessage(content="给我讲个笑话")],
})
print(result["messages"][-1].content)

## 8. 三种构建方式对比

| 方式 | 代码量 | 灵活性 | 适用场景 |
|------|--------|--------|----------|
| 手动构建 | 多 | 最高 | 需要自定义 State、非标准流程 |
| ToolNode + 手动路由 | 中等 | 高 | 标准工具执行 + 自定义路由逻辑 |
| `create_react_agent` | 1 行 | 一般 | 标准 ReAct Agent |

**建议**：
- 先用 `create_react_agent` 快速验证
- 需要更多控制时，退回到手动构建
- 理解手动构建的原理，才能在 `create_react_agent` 不够用时知道怎么改

## 总结

本章学习了：
- ✅ Tool Calling 的完整流程：定义工具 → bind_tools → 模型决策 → 执行工具 → 模型总结
- ✅ `@tool` 装饰器定义工具（类型标注 + docstring）
- ✅ 手动构建 ReAct Agent（理解 model-tools 循环的本质）
- ✅ `ToolNode` 简化工具执行节点
- ✅ `create_react_agent` 一行创建完整 Agent
- ✅ 用 `prompt` 参数控制 Agent 行为

## 下一章预告

**第五章：Human-in-the-Loop（中断、审批、人工干预）** —— 目前 Agent 完全自主运行。下一章学习如何在关键步骤暂停执行，等待人类审批后再继续——这是生产环境中必不可少的安全机制。